# Unsupervised learning: drug-reaction clustering

This notebook groups suspect ingredients that show up with similar reactions. It starts from `data/eda_start/reports_level.parquet`, one row per case. `suspect_ingredients` and `reaction_pts` on that row are ` | `-separated lists, so the first job is to count ingredient-reaction pairs and turn those counts into a profile for each ingredient.

The groups are reporting patterns. FAERS has no count of people who took the drug and did not report anything. Three quarterly files never opened, so 2023 Q3, 2024 Q1, and 2024 Q4 are absent from this table.


## 1. Load the case columns

Only the case id and the two lists are needed. The rest of `reports_level` stays on disk.


In [1]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

for module_name, package_name in (
    ("sklearn", "scikit-learn"),
    ("plotly", "plotly"),
):
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

from collections import Counter
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 40)

REPORTS_LEVEL_PATH = Path("data/eda_start/reports_level.parquet")
CLUSTERS_PATH = Path("data/eda_start/drug_reaction_clusters.parquet")
FIGURE_DIR = Path("reports/figures")
COLUMNS = ["safetyreportid", "suspect_ingredients", "reaction_pts"]
BATCH_SIZE = 200_000
TOP_N = 400
K_RANGE = range(4, 13)
RANDOM_STATE = 0

if not REPORTS_LEVEL_PATH.exists():
    raise FileNotFoundError(
        "data/eda_start/reports_level.parquet is missing. Run notebook 1 first."
    )

parquet = pq.ParquetFile(REPORTS_LEVEL_PATH)
print(f"loaded {REPORTS_LEVEL_PATH}")
print(f"rows = {parquet.metadata.num_rows:,}")


loaded data/eda_start/reports_level.parquet
rows = 8,555,729


## 2. Count pairs in batches

A case with three suspect ingredients and four reactions contributes 12 pairs. Counting those pairs for every case at once would hold a very large frame in memory. Each batch updates three counters and is then dropped:

- how many cases mention each ingredient
- how many cases mention each reaction
- how many cases mention each ingredient-reaction pair, limited to the 400 most common ingredients and the 400 most common reactions

The first pass finds those 400 names. The second pass counts only pairs among them. A case with no suspect ingredient is skipped.


In [2]:
def split_terms(value) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    text = str(value).strip()
    if not text or text.lower() in {"nan", "<na>"}:
        return []
    return [part.strip() for part in text.split(" | ") if part.strip()]


def iter_case_batches():
    for batch in parquet.iter_batches(batch_size=BATCH_SIZE, columns=COLUMNS):
        yield batch.to_pandas()


def count_names() -> tuple[Counter, Counter]:
    ingredients: Counter = Counter()
    reactions: Counter = Counter()
    for frame in iter_case_batches():
        for ingredient_text, reaction_text in zip(frame["suspect_ingredients"], frame["reaction_pts"]):
            ingredient_names = split_terms(ingredient_text)
            if not ingredient_names:
                continue
            ingredients.update(set(ingredient_names))
            reactions.update(set(split_terms(reaction_text)))
    return ingredients, reactions


ingredient_cases, reaction_cases = count_names()
top_ingredients = {name for name, _count in ingredient_cases.most_common(TOP_N)}
top_reactions = {name for name, _count in reaction_cases.most_common(TOP_N)}
print(f"ingredients with at least one case: {len(ingredient_cases):,}")
print(f"reactions with at least one case:   {len(reaction_cases):,}")
print(f"kept {len(top_ingredients)} ingredients and {len(top_reactions)} reactions")


def count_pairs(ingredient_keep: set[str], reaction_keep: set[str]) -> Counter:
    pairs: Counter = Counter()
    for frame in iter_case_batches():
        for ingredient_text, reaction_text in zip(frame["suspect_ingredients"], frame["reaction_pts"]):
            ingredient_names = [name for name in split_terms(ingredient_text) if name in ingredient_keep]
            reaction_names = [name for name in split_terms(reaction_text) if name in reaction_keep]
            if not ingredient_names or not reaction_names:
                continue
            pairs.update(product(set(ingredient_names), set(reaction_names)))
    return pairs


pair_counts = count_pairs(top_ingredients, top_reactions)
print(f"ingredient-reaction pairs kept: {len(pair_counts):,}")


ingredients with at least one case: 11,477
reactions with at least one case:   21,654
kept 400 ingredients and 400 reactions
ingredient-reaction pairs kept: 145,320


## 3. Reaction profiles

Each row of the matrix is one ingredient. Each column is one reaction. The entry is how many cases listed that pair. Dividing the row by its total makes a profile: two drugs can cluster together when they share a reaction pattern, even if one is reported much more often.

An ingredient whose reactions all fall outside the 400 most common reactions has an empty row and is left out of the fit.


In [3]:
ingredient_names = sorted(top_ingredients)
reaction_names = sorted(top_reactions)
count_matrix = pd.DataFrame(0, index=ingredient_names, columns=reaction_names, dtype="int32")
for (ingredient, reaction), count in pair_counts.items():
    count_matrix.at[ingredient, reaction] = count

row_totals = count_matrix.sum(axis=1)
count_matrix = count_matrix.loc[row_totals > 0]
row_totals = row_totals.loc[count_matrix.index]
profiles = count_matrix.div(row_totals, axis=0)
print("profile matrix:", profiles.shape)


profile matrix: (400, 400)


## 4. Choose the number of clusters

K-means is fit for every k from 4 to 12. Inertia falls as k grows. The elbow is the k farthest from a straight line drawn between the first and last inertia. Silhouette is computed on a sample of ingredients and is higher when clusters are compact and separated. The fit below uses the k with the higher silhouette. The elbow k is printed beside it.


In [4]:
def elbow_k(ks: list[int], inertias: list[float]) -> int:
    x = np.asarray(ks, dtype=float)
    y = np.asarray(inertias, dtype=float)
    start = np.array([x[0], y[0]])
    end = np.array([x[-1], y[-1]])
    line = end - start
    line_norm = np.linalg.norm(line)
    distances = []
    for xi, yi in zip(x, y):
        point = np.array([xi, yi])
        distance = np.linalg.norm(np.cross(line, start - point)) / line_norm
        distances.append(distance)
    return ks[int(np.argmax(distances))]


profile_values = profiles.to_numpy(dtype=float)
sample_size = min(200, len(profiles))
sample_index = np.random.default_rng(RANDOM_STATE).choice(len(profiles), size=sample_size, replace=False)

k_rows = []
for k in K_RANGE:
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(profile_values)
    score = silhouette_score(profile_values[sample_index], labels[sample_index])
    k_rows.append({"k": k, "inertia": model.inertia_, "silhouette": score})

k_table = pd.DataFrame(k_rows)
elbow = elbow_k(k_table["k"].tolist(), k_table["inertia"].tolist())
chosen = k_table.sort_values(["silhouette", "k"], ascending=[False, True]).iloc[0]
chosen_k = int(chosen["k"])
display(k_table.round(4))
print(f"elbow k = {elbow}")
print(f"chosen k = {chosen_k}  silhouette = {chosen['silhouette']:.4f}")


/tmp/ipykernel_46/4054079926.py:11: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  distance = np.linalg.norm(np.cross(line, start - point)) / line_norm


,k,inertia,silhouette
0,4,5.8727,0.4259
1,5,5.5150,0.1190
2,6,5.2285,0.1419
3,7,5.0559,0.1498
4,8,4.7463,0.1299
5,9,4.6334,0.0914
6,10,4.4552,0.1378
7,11,4.3674,0.1075
8,12,4.1783,0.1002


elbow k = 8
chosen k = 4  silhouette = 0.4259


## 5. What is in each cluster

The final k-means uses the chosen k. For each cluster the notebook prints how many ingredients it holds, the ingredients reported in the most cases, and the reactions those ingredients are most often listed with.


In [5]:
final_model = KMeans(n_clusters=chosen_k, n_init=10, random_state=RANDOM_STATE)
cluster_labels = final_model.fit_predict(profile_values)
assignments = pd.DataFrame(
    {"ingredient": profiles.index, "cluster": cluster_labels},
)
assignments["n_cases"] = assignments["ingredient"].map(ingredient_cases)

cluster_rows = []
for cluster_id, group in assignments.groupby("cluster"):
    members = group.sort_values("n_cases", ascending=False)
    member_names = members["ingredient"].tolist()
    reaction_totals = count_matrix.loc[member_names].sum(axis=0).sort_values(ascending=False)
    top_reactions = " | ".join(reaction_totals.head(8).index.tolist())
    top_ingredients = " | ".join(member_names[:8])
    cluster_rows.append(
        {
            "cluster": int(cluster_id),
            "n_ingredients": len(member_names),
            "top_ingredients": top_ingredients,
            "top_reactions": top_reactions,
        }
    )
    print(f"cluster {cluster_id}: {len(member_names)} ingredients")
    print(f"  ingredients: {top_ingredients}")
    print(f"  reactions:   {top_reactions}")

cluster_summary = pd.DataFrame(cluster_rows).sort_values("cluster")
display(cluster_summary)


cluster 0: 380 ingredients
  ingredients: DUPILUMAB | RANITIDINE HYDROCHLORIDE | ADALIMUMAB | LENALIDOMIDE | RANITIDINE | TIRZEPATIDE | RITUXIMAB | METHOTREXATE
  reactions:   DRUG INEFFECTIVE | OFF LABEL USE | FATIGUE | PAIN | NAUSEA | DIARRHOEA | DYSPNOEA | RASH
cluster 1: 4 ingredients
  ingredients: LATANOPROST | BIMATOPROST | BRIMONIDINE TARTRATE | OLOPATADINE HYDROCHLORIDE
  reactions:   TREATMENT FAILURE | DRUG INEFFECTIVE | EYE IRRITATION | HYPERSENSITIVITY | OCULAR HYPERAEMIA | VISION BLURRED | EYE PAIN | EYE PRURITUS
cluster 2: 1 ingredients
  ingredients: MEDROXYPROGESTERONE ACETATE
  reactions:   MENINGIOMA | DRUG DOSE OMISSION BY DEVICE | OFF LABEL USE | DEVICE LEAKAGE | INCORRECT DOSE ADMINISTERED BY DEVICE | NEEDLE ISSUE | SYRINGE ISSUE | DEVICE ISSUE
cluster 3: 15 ingredients
  ingredients: OXYCODONE HYDROCHLORIDE | ACETAMINOPHEN\OXYCODONE HYDROCHLORIDE | ACETAMINOPHEN\HYDROCODONE BITARTRATE | MORPHINE SULFATE | HYDROMORPHONE HYDROCHLORIDE | BUPRENORPHINE | TRAMADOL | F

,cluster,n_ingredients,top_ingredients,top_reactions
0,0,380,DUPILUMAB | RANITIDINE HYDROCHLORIDE | ADALIMUMAB | LENALIDOMIDE | RANITIDIN...,DRUG INEFFECTIVE | OFF LABEL USE | FATIGUE | PAIN | NAUSEA | DIARRHOEA | DYS...
1,1,4,LATANOPROST | BIMATOPROST | BRIMONIDINE TARTRATE | OLOPATADINE HYDROCHLORIDE,TREATMENT FAILURE | DRUG INEFFECTIVE | EYE IRRITATION | HYPERSENSITIVITY | O...
2,2,1,MEDROXYPROGESTERONE ACETATE,MENINGIOMA | DRUG DOSE OMISSION BY DEVICE | OFF LABEL USE | DEVICE LEAKAGE |...
3,3,15,OXYCODONE HYDROCHLORIDE | ACETAMINOPHEN\OXYCODONE HYDROCHLORIDE | ACETAMINOP...,DRUG DEPENDENCE | OVERDOSE | PAIN | EMOTIONAL DISTRESS | DRUG WITHDRAWAL SYN...


## 6. Ingredient-reaction map

Each band is one cluster. Ingredients are on the left and reactions are on the right. A line means that reaction is one of the ingredient's most common reactions. Cluster 0 shows its eight most-reported ingredients. The other clusters show every ingredient.

In [6]:
palette = px.colors.qualitative.Plotly
key_frames = []
reaction_clusters = {}
for cluster_id, group in assignments.groupby("cluster"):
    cluster_id = int(cluster_id)
    ranked = group.sort_values("n_cases", ascending=False)
    shown = ranked.head(8) if cluster_id == 0 else ranked
    key_frames.append(shown.assign(cluster=cluster_id))
    totals = count_matrix.loc[ranked["ingredient"].tolist()].sum(axis=0)
    totals = totals[totals > 0].sort_values(ascending=False).head(8)
    for reaction in totals.index:
        reaction_clusters.setdefault(reaction, set()).add(cluster_id)

key_ingredients = pd.concat(key_frames, ignore_index=True)
displayed_reactions = set(reaction_clusters)

links = []
for row in key_ingredients.itertuples(index=False):
    scores = count_matrix.loc[row.ingredient]
    scores = scores[scores > 0].sort_values(ascending=False).head(8)
    for reaction, count in scores.items():
        if reaction not in displayed_reactions:
            continue
        share = float(profiles.loc[row.ingredient, reaction])
        links.append(
            {
                "ingredient": row.ingredient,
                "cluster": int(row.cluster),
                "n_cases": int(row.n_cases),
                "reaction": reaction,
                "count": int(count),
                "share": share,
            }
        )

exclusive = {
    reaction: next(iter(clusters))
    for reaction, clusters in reaction_clusters.items()
    if len(clusters) == 1
}
shared = [reaction for reaction, clusters in reaction_clusters.items() if len(clusters) > 1]

band_top = {}
cursor = 0.0
gap = 1.6
for cluster_id in sorted(key_ingredients["cluster"].unique()):
    cluster_id = int(cluster_id)
    n_ingredients = int((key_ingredients["cluster"] == cluster_id).sum())
    n_reactions = sum(owner == cluster_id for owner in exclusive.values())
    slots = max(n_ingredients, n_reactions, 1)
    band_top[cluster_id] = cursor
    cursor -= slots - 1 + gap


def centered(n_items: int, n_slots: int, top: float) -> list[float]:
    if n_items <= 0:
        return []
    if n_items == 1:
        return [top - (n_slots - 1) / 2]
    span = n_slots - 1
    offset = (span - (n_items - 1)) / 2
    return [top - (offset + i) for i in range(n_items)]


ingredient_y = {}
reaction_y = {}
for cluster_id, top in band_top.items():
    names = (
        key_ingredients.loc[key_ingredients["cluster"] == cluster_id]
        .sort_values("n_cases", ascending=False)["ingredient"]
        .tolist()
    )
    owned = [reaction for reaction, owner in exclusive.items() if owner == cluster_id]
    owned = sorted(owned, key=lambda name: -float(count_matrix[name].sum()))
    slots = max(len(names), len(owned), 1)
    for name, y in zip(names, centered(len(names), slots, top)):
        ingredient_y[name] = y
    for name, y in zip(owned, centered(len(owned), slots, top)):
        reaction_y[name] = y

cluster_of = key_ingredients.set_index("ingredient")["cluster"].astype(int)
band_center = {
    cluster_id: float(np.mean([y for name, y in ingredient_y.items() if cluster_of.loc[name] == cluster_id]))
    for cluster_id in band_top
}
occupied = list(reaction_y.values())
for reaction in sorted(shared, key=lambda name: float(np.mean([band_center[c] for c in reaction_clusters[name]]))):
    y = float(np.mean([band_center[c] for c in reaction_clusters[reaction]]))
    while any(abs(y - other) < 0.55 for other in occupied):
        y -= 0.55
    reaction_y[reaction] = y
    occupied.append(y)

ingredient_x = 0.0
reaction_x = 1.0
by_ingredient = {}
by_reaction = {}
for link in links:
    by_ingredient.setdefault(link["ingredient"], []).append(link)
    by_reaction.setdefault(link["reaction"], []).append(link)


def ingredient_hover(name: str) -> str:
    row = key_ingredients.set_index("ingredient").loc[name]
    lines = [
        f"<b>{name}</b>",
        f"cluster {int(row['cluster'])}",
        f"cases: {int(row['n_cases']):,}",
        "reactions:",
    ]
    for link in by_ingredient.get(name, []):
        lines.append(f"{link['reaction']} ({link['share']:.1%} of profile)")
    if len(lines) == 4:
        lines[-1] = "reactions: none of this ingredient's leading reactions are on the map"
    return "<br>".join(lines)


def reaction_hover(name: str) -> str:
    lines = [f"<b>{name}</b>", "ingredients:"]
    partners = sorted(by_reaction.get(name, []), key=lambda link: (link["cluster"], link["ingredient"]))
    if not partners:
        lines.append("no key ingredient on the map lists this among its leading reactions")
    for link in partners:
        lines.append(f"{link['ingredient']} (cluster {link['cluster']})")
    return "<br>".join(lines)


def most_connected(link_map: dict[str, list], limit: int = 5) -> set[str]:
    ranked = sorted(link_map, key=lambda name: (-len(link_map[name]), name))
    if not ranked:
        return set()
    threshold = len(link_map[ranked[min(limit, len(ranked)) - 1]])
    return {name for name in ranked if len(link_map[name]) >= threshold}


bold_ingredients = most_connected(by_ingredient)
bold_reactions = most_connected(by_reaction)
print("most connected ingredients")
for name in sorted(bold_ingredients, key=lambda item: (-len(by_ingredient[item]), item)):
    print(f"  {len(by_ingredient[name])} links  cluster {int(cluster_of.loc[name])}  {name}")
print("most connected reactions")
for name in sorted(bold_reactions, key=lambda item: (-len(by_reaction[item]), item)):
    print(f"  {len(by_reaction[name])} links  {name}")

relationship_map = go.Figure()
for cluster_id in sorted(band_top):
    color = palette[cluster_id % len(palette)]
    cluster_links = [link for link in links if link["cluster"] == cluster_id]
    edge_x = []
    edge_y = []
    mid_x = []
    mid_y = []
    edge_hover = []
    for link in cluster_links:
        y0 = ingredient_y[link["ingredient"]]
        y1 = reaction_y[link["reaction"]]
        edge_x.extend([ingredient_x, reaction_x, None])
        edge_y.extend([y0, y1, None])
        mid_x.append((ingredient_x + reaction_x) / 2)
        mid_y.append((y0 + y1) / 2)
        edge_hover.append(
            f"{link['ingredient']} \u2014 {link['reaction']}<br>reports: {link['count']:,}<br>share of profile: {link['share']:.1%}"
        )
    relationship_map.add_trace(
        go.Scatter(
            x=edge_x,
            y=edge_y,
            mode="lines",
            line={"color": color, "width": 1.4},
            hoverinfo="skip",
            legendgroup=str(cluster_id),
            showlegend=False,
        )
    )
    relationship_map.add_trace(
        go.Scatter(
            x=mid_x,
            y=mid_y,
            mode="markers",
            marker={"size": 14, "color": color, "opacity": 0.01},
            hovertext=edge_hover,
            hoverinfo="text",
            legendgroup=str(cluster_id),
            showlegend=False,
        )
    )
    names = (
        key_ingredients.loc[key_ingredients["cluster"] == cluster_id]
        .sort_values("n_cases", ascending=False)["ingredient"]
        .tolist()
    )
    legend_drawn = False
    for highlighted, group_names in (
        (False, [name for name in names if name not in bold_ingredients]),
        (True, [name for name in names if name in bold_ingredients]),
    ):
        if not group_names:
            continue
        relationship_map.add_trace(
            go.Scatter(
                x=[ingredient_x] * len(group_names),
                y=[ingredient_y[name] for name in group_names],
                mode="markers+text",
                marker={
                    "size": 18 if highlighted else 11,
                    "color": color,
                    "line": {"width": 2 if highlighted else 0, "color": "#111111"},
                },
                text=group_names,
                textposition="middle left",
                textfont={
                    "size": 14 if highlighted else 11,
                    "family": "Arial Black" if highlighted else "Arial",
                },
                hovertext=[ingredient_hover(name) for name in group_names],
                hoverinfo="text",
                name=f"cluster {cluster_id}",
                legendgroup=str(cluster_id),
                showlegend=not legend_drawn,
            )
        )
        legend_drawn = True

reaction_names = sorted(reaction_y, key=reaction_y.get, reverse=True)
legend_drawn = False
for highlighted, group_names in (
    (False, [name for name in reaction_names if name not in bold_reactions]),
    (True, [name for name in reaction_names if name in bold_reactions]),
):
    if not group_names:
        continue
    relationship_map.add_trace(
        go.Scatter(
            x=[reaction_x] * len(group_names),
            y=[reaction_y[name] for name in group_names],
            mode="markers+text",
            marker={
                "size": 16 if highlighted else 9,
                "color": "#111111" if highlighted else "#333333",
                "symbol": "diamond",
                "line": {"width": 1 if highlighted else 0, "color": "#111111"},
            },
            text=group_names,
            textposition="middle right",
            textfont={
                "size": 14 if highlighted else 11,
                "family": "Arial Black" if highlighted else "Arial",
            },
            hovertext=[reaction_hover(name) for name in group_names],
            hoverinfo="text",
            name="reaction",
            legendgroup="reaction",
            showlegend=not legend_drawn,
        )
    )
    legend_drawn = True

y_values = list(ingredient_y.values()) + list(reaction_y.values())
span = max(y_values) - min(y_values)
relationship_map.update_layout(
    title="Key ingredients and the reactions reported with them",
    xaxis={"visible": False, "range": [-0.62, 1.72]},
    yaxis={"visible": False, "range": [min(y_values) - 0.8, max(y_values) + 1.1]},
    height=int(max(720, span * 42 + 180)),
    width=1180,
    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode="closest",
    margin={"l": 20, "r": 20, "t": 60, "b": 20},
    legend_title_text="cluster",
)
relationship_map.add_annotation(
    x=ingredient_x, y=max(y_values) + 0.7, text="ingredients", showarrow=False, font={"size": 13}
)
relationship_map.add_annotation(
    x=reaction_x, y=max(y_values) + 0.7, text="reactions", showarrow=False, font={"size": 13}
)
for cluster_id, top in band_top.items():
    relationship_map.add_annotation(
        x=-0.58,
        y=band_center[cluster_id],
        text=f"cluster {cluster_id}",
        showarrow=False,
        font={"size": 12, "color": palette[cluster_id % len(palette)]},
        xanchor="left",
    )

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
relationship_map.write_html(FIGURE_DIR / "clustering_relationship_map.html", include_plotlyjs=True)
relationship_map.show()


most connected ingredients
  8 links  cluster 3  BUPRENORPHINE
  8 links  cluster 2  MEDROXYPROGESTERONE ACETATE
  8 links  cluster 1  OLOPATADINE HYDROCHLORIDE
  7 links  cluster 1  BRIMONIDINE TARTRATE
  7 links  cluster 3  FENTANYL
  7 links  cluster 3  HYDROMORPHONE
  7 links  cluster 1  LATANOPROST
  7 links  cluster 3  MORPHINE SULFATE
  7 links  cluster 3  TRAMADOL
most connected reactions
  17 links  PAIN
  15 links  DRUG DEPENDENCE
  15 links  OVERDOSE
  14 links  EMOTIONAL DISTRESS
  12 links  DRUG INEFFECTIVE
  12 links  DRUG WITHDRAWAL SYNDROME


The bold, larger names are the nodes with the most lines. Nine ingredients are tied at the top, each with at least seven links: buprenorphine, fentanyl, hydromorphone, morphine sulfate, and tramadol in cluster 3; brimonidine tartrate, latanoprost, and olopatadine hydrochloride in cluster 1; and medroxyprogesterone acetate in cluster 2. Their own leading reactions are the ones drawn for those bands. None of cluster 0's eight most-reported ingredients reach that count. Dupilumab has two such links, and both ranitidine names have none, because their leading reactions are more specific than the eight reactions drawn for that large band.

The bold reactions are pain (17 links), drug dependence and overdose (15), emotional distress (14), and drug ineffective and drug withdrawal syndrome (12). Pain and drug ineffective collect lines from more than one band, so they are shared reporting language. Dependence, overdose, emotional distress, and withdrawal sit in the opioid band, which is why so many of those ingredients connect to them.

Cluster 0's band is still only its eight most-reported ingredients. A high link count means those reactions were among the ones drawn for these clusters. These are reporting links, and they leave out how often a drug is taken.

## 7. Saved ingredient table

`drug_reaction_clusters.parquet` has one row per ingredient that entered the matrix. `top_reactions` lists the reactions most often reported with that ingredient, not the reactions that merely define the cluster.


In [7]:
def ingredient_top_reactions(ingredient: str) -> str:
    ranked = count_matrix.loc[ingredient].sort_values(ascending=False)
    ranked = ranked[ranked > 0].head(8)
    if ranked.empty:
        return ""
    return " | ".join(ranked.index.tolist())


clusters = assignments.copy()
clusters["top_reactions"] = clusters["ingredient"].map(ingredient_top_reactions)
clusters = clusters[["ingredient", "cluster", "top_reactions"]].sort_values(["cluster", "ingredient"])
CLUSTERS_PATH.parent.mkdir(parents=True, exist_ok=True)
clusters.to_parquet(CLUSTERS_PATH, index=False)
print(f"saved {CLUSTERS_PATH}  rows={len(clusters):,}")
display(clusters.head(12))


saved data/eda_start/drug_reaction_clusters.parquet  rows=400


,ingredient,cluster,top_reactions
0,ABALOPARATIDE,0,HEADACHE | NAUSEA | DIZZINESS | FATIGUE | PRODUCT DOSE OMISSION ISSUE | HEAR...
1,ABATACEPT,0,DRUG INEFFECTIVE | RHEUMATOID ARTHRITIS | ARTHRALGIA | JOINT SWELLING | PAIN...
2,ABEMACICLIB,0,DIARRHOEA | FATIGUE | NAUSEA | MALIGNANT NEOPLASM PROGRESSION | VOMITING | D...
3,ABIRATERONE ACETATE,0,DEATH | FATIGUE | DRUG INEFFECTIVE | HOSPITALISATION | DISEASE PROGRESSION |...
4,ACALABRUTINIB,0,DEATH | FATIGUE | HEADACHE | PRODUCT DOSE OMISSION ISSUE | FALL | DIARRHOEA ...
5,ACETAMINOPHEN,0,DRUG INEFFECTIVE | TOXICITY TO VARIOUS AGENTS | PAIN | OFF LABEL USE | VOMIT...
6,ACETAMINOPHEN\CODEINE PHOSPHATE,0,DEPENDENCE | OVERDOSE | PAIN | DRUG DEPENDENCE | OFF LABEL USE | FATIGUE | D...
10,ACYCLOVIR,0,DRUG INEFFECTIVE | ACUTE KIDNEY INJURY | OFF LABEL USE | CONDITION AGGRAVATE...
11,ADALIMUMAB,0,DRUG INEFFECTIVE | PAIN | ARTHRALGIA | RHEUMATOID ARTHRITIS | FATIGUE | OFF ...
12,ADALIMUMAB-ADAZ,0,OFF LABEL USE | DRUG INEFFECTIVE | ARTHRALGIA | FATIGUE | PAIN | DIARRHOEA |...


## 8. How to read the result

Ingredients in the same cluster are reported with similar reaction lists. That is not a finding that the drugs cause those reactions, and it is not a measure of how risky a drug is. A widely used drug produces more reports even when the chance of a reaction is small.

The case table is also missing 2023 Q3, 2024 Q1, and 2024 Q4. Replacing those zips and rebuilding `reports_level.parquet` will change these counts.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=73434ccc-8a09-4578-aac6-0389f491d707' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>